In [ ]:

from minio import Minio
from minio.error import S3Error
import urllib3
from util import add_ah_throughput


from util.add_ah_throughput import add_ah_throughput

minio_endpoint = "optimusprime.isea.rwth-aachen.de:9000"
access_key= "8ms0O8n4gwMia5BpDrYq"
secret_key= "WxDnJMQmUrW8RScViRf0CCTBDDIlZaoIANQgTFWl"
bucket_name= "zho"
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

if access_key and secret_key and minio_endpoint:
    minio_client = Minio(
        minio_endpoint,
        access_key=access_key,
        secret_key=secret_key,
        secure=True,
        cert_check=False,
    )
type_cell = 'VTC'

objects = minio_client.list_objects(
    bucket_name=bucket_name,
    prefix=f"Metabatt/{type_cell}/",
    recursive=True,
)

folders = set()
prefix = "Metabatt/{type_cell}/"

for obj in objects:
    # Remove the prefix
    path_after_prefix = obj.object_name[len(prefix):]
    
    # Get the first folder name
    if '/' in path_after_prefix:
        folder_name = path_after_prefix.split('/')[0]
        folders.add(folder_name)


def get_ah_throughput_for_cell_from_s3(cell):
    objects = minio_client.list_objects(
        bucket_name=bucket_name,
        prefix=f"Metabatt/{type_cell}/{cell}/",
        recursive=True,
    )
    # Get list of parquet files
    cell_tests = [obj.object_name for obj in objects if obj.object_name.endswith('.parquet')]
    
    if any("jri_CU" in test for test in cell_tests):
        tests = []
        try:
            for test_file in cell_tests:                
                # Get file from MinIO
                response = minio_client.get_object(bucket_name, test_file)
                data = response.read()
                response.close()
                response.release_conn()
                desired_columns = ['Zeit', 'Strom']
                columns_to_read = [col for col in desired_columns if col in desired_columns]
                if columns_to_read == []:
                    print(f"Skipping {test_file} due to missing columns")
                    continue  # Skip if none of the desired columns are present
                df = pd.read_parquet(io.BytesIO(data), columns=columns_to_read)
                df = df.drop_duplicates('Zeit')
                tests.append(df)

            df_cell = pd.concat(tests, ignore_index=True)

            # Fix the format
            df_cell[df_cell.select_dtypes(np.float64).columns] = df_cell.select_dtypes(
                np.float64
            ).astype(np.float32)
            df_cell = df_cell.rename(
                columns={
                    "Strom": "Current",
                    "Zeit": "Time",
                }
            )
            df_cell["Time_UTC"] = df_cell.Time.dt.tz_convert("UTC")

            return add_ah_throughput(df_cell)
        
        except S3Error as e:
            print("Error occurred.", e)


In [ ]:
import glob
import io
import os
import pandas as pd
import numpy as np

working_path = f'Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten\Checkup-Parquet\{type_cell}'
target_specimen = None

List_Cell = glob.glob1(working_path, '*.parquet')

Cell_subset = [
    items
    for items in List_Cell
    if not target_specimen or any(target in items for target in target_specimen)
]


In [ ]:
for cell in Cell_subset:
    savepath_df_GOLD = os.path.join(
    working_path,  cell
    )
    if (os.path.exists(savepath_df_GOLD)):
        print(f"Processing cell: {cell}")
        try:
            df_cell = get_ah_throughput_for_cell_from_s3(cell.split('.')[0])

            df_gold = pd.read_parquet(savepath_df_GOLD)
            df_gold_Ah_throughput = df_gold.drop(columns=["Ah_throughput"], axis=1).drop_duplicates('Time')
            df_gold_Ah_throughput['Ah_throughput']= df_gold_Ah_throughput['Time'].map(df_cell.set_index('Time')['Ah_throughput'])        
            df_gold_Ah_throughput.to_parquet(savepath_df_GOLD, index=False)
            print(f"Updated Ah_throughput for cell: {cell}")
        except Exception as e:
            print(f"Error found: {e}")

In [ ]:
df_cell 

In [ ]:
df_cell